In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np, os
import matplotlib.pyplot as plt

from pathlib import Path
import yaml
import re

import pyarrow as pa
import pyarrow.parquet as pq

import vivarium.gbd_mapping as gm
from vivarium.artifact import Artifact

# from db_queries import get_ids, get_outputs, get_population, get_covariate_estimates
# from get_draws.api import get_draws

import vivarium_helpers as vh
# import vivarium_helpers.id_helper as idh
from vivarium_helpers.vph_output.operations import VPHOperator
from vivarium_helpers.vph_output import cleaning

!date
!whoami
!uname -a
!pwd

# V&V List


1. Positive BBBM tests result in treatment initiation rates that match
    the year/location specific rates from in the [treatment intervention
    data table](https://vivarium-research.readthedocs.io/en/latest/models/intervention_models/alzheimers/hypothetical_treatment.html#alzheimers-intervention-treatment-data-table)

1. 10% of transitions to Full treatment effect status are by simulants who discontinue treatment

1. Full/Waning durations are accurate (use person-time ratios between states?)

1. “In treatment/waiting for treatment” duration is accurate (use person-time ratios between states?)

1.  Interactive sim verification spot checking a simulant’s durations in
    treatment statuses as they move through BBBM test negative, Full
    treatment effect, Waning treatment effect, No treatment effect
    statuses (for both completed and discontinued treatments)

1.  Check hazard ratios for simulants who begin treatment and those who
    transition to *No treatment effect* (Check hazard ratio of treated to
    not treated for hazard rate from BBBM to MCI)

Time- and location-specific treatment initiation rate from treatment
intervention data table:

> Lilly: “The percent of patients with a positive BBBM test who initiate treatment will vary by location and over time – but will not vary by age or sex. In the US: 30% of eligible patients initiate (constant 2030-2100); Japan: 80% of eligible patients initiate (constant 2030-2100); all other countries: 40% of eligible patients initiate in 2030, increasing linearly to 70% by 2035, remaining constant at 70% until 2100.””

# Find data

In [ ]:
# Project directory on the cluster
project_dir = '/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/'

if os.sys.platform == 'darwin':
    # Notebook is running on Nathaniel's MacBook
    local_project_dir = "/Users/ndbs/project_data/vivarium_csu_alzheimers/"
else:
    # Assume we're running on the cluster
    local_project_dir = project_dir

In [ ]:
# Project directory
%cd $local_project_dir

In [ ]:
# Results directory containing model 15.0 results for three locations
model_run_subdir = 'results/model15.0/model_spec/2026_11_13_15_13_04/'
!ls -halt $model_run_subdir/results

In [ ]:
locations = [
    'United States of America',
    'Brazil',
    'China',
]

# Define some shorter names to use for plotting
location_to_short_name = ({loc: loc for loc in locations} | {
    'United Kingdom': 'UK',
    'United States of America': 'USA',
})

# # Select a subset of locations to draw plots for
# locations_to_plot = locations[:2]

# project_dir = '/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/'

artifact_model_number = '13.0' # Artifacts are stored here
run_subdirectories = [
    model_run_subdir,
]
# Where results are stored locally
run_dirs = [local_project_dir + run_subdir for run_subdir in run_subdirectories]
results_dirs = [run_dir + 'results/' for run_dir in run_dirs]

# # Option 1: One results directory per location
# location_to_results_dir = {
#     loc: path for loc, path in zip(locations, results_dirs)}

# Option 2: All locations in one results directory
location_to_results_dir = {'all': results_dirs[0]}

location_to_artifact_subdir = {loc: loc.lower().replace(' ', '_') for loc in locations}
artifact_subpaths = [f'artifacts/model{artifact_model_number}/' + subdir + '.hdf' for subdir in location_to_artifact_subdir.values()]
# Map locations to where artifacts are stored locally (for reading artifact data)
location_to_local_artifact_path = {
    loc: local_project_dir + subpath for loc, subpath in zip(locations, artifact_subpaths)}
# Map locations to where artifacts are stored on the cluster (for adding locations to sim results)
location_to_artifact_path = {loc: project_dir + subpath for loc, subpath in zip(locations, artifact_subpaths)}
location_to_artifact_path

In [ ]:
location_to_results_dir

# Define some ordered Categorical dtypes, and convert years to ints

In [ ]:
# # Instead of creating this dtype here, create an appropriate dtype in
# # the load_sim_output function.
# # Order locations lexicographically
# all_locations = [
#     'United States of America',
#     'Brazil',
#     'China',
#     'Germany',
#     'Israel',
#     'Japan',
#     'Spain',
#     'Sweden',
#     'Taiwan',
#     'United Kingdom',
# ]
# location_dtype = pd.CategoricalDtype(sorted(all_locations), ordered=True)

# Order years chronologically (for some reason years are strings in the output)
# years = [str(year) for year in range(2025, 2101)]
# year_dtype = pd.CategoricalDtype(years, ordered=True)
# int16 ranges from -32768 to 32767 (I think), which is sufficient to
# represent all years 2025-2100. uint8 only goes from 0 to 255, which is
# too small.
year_dtype = 'int16'

# Order age groups chronologically
age_groups = [f'{age}_to_{age + 4}' for age in range(25, 95, 5)] + ['95_plus']
age_group_dtype = pd.CategoricalDtype(age_groups, ordered=True)

# Order scenarios by complexity. Each testing speed has a scenario with
# zero impact treatment and a scenario with treatment.
testing_speeds = ['slow', 'intermediate', 'accelerated']
zero_impact_scenarios = [
    f'{speed}_testing_zero_impact_treatment' for speed in testing_speeds]
with_treatment_scenarios = [
    f'{speed}_testing_with_treatment' for speed in testing_speeds]
scenarios = ['baseline'] + zero_impact_scenarios + with_treatment_scenarios
scenario_dtype = pd.CategoricalDtype(scenarios, ordered=True)

colname_to_dtype = {
    # 'location': location_dtype,
    'event_year': year_dtype,
    'age_group': age_group_dtype,
    'scenario': scenario_dtype,
}

# Load one artifact and define age bins

In [ ]:
artifact_path = location_to_local_artifact_path[locations[0]]
art = Artifact(artifact_path)
print(art.load('metadata.locations'))

In [ ]:
# age_bins is an empty DataFrame with a MultiIndex storing age group data
age_bins = art.load('population.age_bins')
age_dictionary = (
    age_bins
    .reset_index()
    .assign(age_group=lambda df: df['age_group_name'].str.replace(' ', '_'))
    # Filter to ages that actually appear in our sim
    .query("age_start >= 25")
)
age_dictionary

# See what the treatment relative risk looks like

Draws should be uniformly distributed in $[0.4, 0.6]$.

In [ ]:
treatment_rr = art.load('treatment.relative_risk')
treatment_rr

# Define functions to load and merge Artifact data from all locations, and to summarize Artifact data for plotting

In [ ]:
def load_artifact_data(
    key,
    filter_terms=None,
    location_to_artifact_path=location_to_artifact_path,
):
    dfs = {} # dict to map locations to artifact data
    for location, path in location_to_artifact_path.items():
        art = Artifact(path, filter_terms)
        # Check to make sure location matches artifact
        art_locations = art.load('metadata.locations')
        assert len(art_locations) == 1 and art_locations[0] == location, \
            f'Unexpected locations in artifact: {location=}, {art_locations=}'
        df = art.load(key)
        dfs[location] = df
    data = pd.concat(dfs, names=['location', *df.index.names])
    return data

def lower(x):
    return x.quantile(0.025)

def upper(x):
    return x.quantile(0.975)

def summarize_artifact_data(df):
    summary = df.agg(['mean', lower, upper], axis='columns')
    return summary

# Define functions to load simulation output and summarize it for plotting

With the `pyarrow` engine, `pandas.read_parquet()` calls `pyarrow.parquet.read_table()` and
`pyarrow.Table.to_pandas()`. Additional keyword arguments passed to
`pandas.read_parquet()` are passed to `pyarrow.parquet.read_table()`.

https://arrow.apache.org/docs/python/generated/pyarrow.parquet.read_table.html#pyarrow.parquet.read_table

https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_pandas

Pyarrow Dictionary arrays are analogous to pandas Categoricals:

https://arrow.apache.org/docs/python/data.html#data-dictionary

Pyarrow data types:

https://arrow.apache.org/docs/python/api/datatypes.html

https://arrow.apache.org/docs/python/generated/pyarrow.types.is_floating.html


Google AI results for "find column names in parquet file":

```python
import pyarrow.parquet as pq

# Replace 'your_file.parquet' with the actual path to your Parquet file
parquet_file_path = 'your_file.parquet'

# Read the Parquet file's schema
parquet_schema = pq.read_schema(parquet_file_path)

# Extract column names
column_names = parquet_schema.names

print(f"Column names in '{parquet_file_path}':")
for name in column_names:
    print(name)
```

In [ ]:
# Create an operator object - treat each random seed as a separate draw,
# and add location to the index
ops = VPHOperator(location_col=True)
# ops.index_cols.extend(['location', 'random_seed'])

def load_sim_output(
        measure,
        results_dict=location_to_results_dir,
        # Pass None to skip filtering locations (when None, must also
        # pass assign_location=False or raw=True)
        location_to_artifact_path=location_to_artifact_path,
        # specify dtypes of certain columns
        colname_to_dtype=colname_to_dtype,
        drop_superfluous_cols=True, # drop redundant or empty columns
        # Sets the 'read_dictionary' key of kwargs, which is passed to
        # pyarrow.parquet.read_table()
        force_parquet_dictionaries=True,
        force_pandas_categoricals=True,
        aggregate_seeds=True,
        assign_location=True,
        raw=False, # Overrides other parameters if True
        **kwargs, # keyword args to pass to .read_parquet
    ):
    """Load simulation output from .parquet files for all locations,
    optionally reducing the size of the data when possible. Returns
    concatenated outputs with a 'location' column added.
    """
    # Override optional transformations if raw=True
    if raw:
        drop_superfluous_cols = False
        force_parquet_dictionaries = False
        force_pandas_categoricals = False
        aggregate_seeds = False
        assign_location = False

    # Determine whether results for all locations are stored in same
    # directory, or if different locations have different results
    # directories
    match location_to_results_dir:
        case {'all': _}:
            all_locations_together = True
        case _:
            all_locations_together = False
    
    if all_locations_together and assign_location and location_to_artifact_path is None:
        raise ValueError(
            "Must provide mapping of artifacts to locations  when" \
            " assign_location=True and all locations are concatenated" \
            " in the simulation outputs."
        )

    dfs = []
    for location, directory in results_dict.items():

        # NOTE: New Vivarium version stores each measure in a folder
        # instead of a single .parquet file
        parquet_file_path = Path(directory) / measure #f'{measure}.parquet'
        # The ParquetDataset object has a .schema attribute we can use
        # to get its column names and data types 
        parquet_dataset = pa.parquet.ParquetDataset(parquet_file_path)
        # # NOTE: This code no longer works with datasets stored in
        # # directories instead of single parquet files:
        # # Read the Parquet file's schema to get column names and data types
        # parquet_schema = pq.read_schema(parquet_file_path)

        if (
            all_locations_together
            and location_to_artifact_path is not None
        ):
            if 'artifact_path' in parquet_dataset.schema.names:
                # Filter to locations in list
                location_filter = (
                    'artifact_path',
                    'in',
                    list(location_to_artifact_path.values()),
                )
                user_filters = kwargs.get('filters') # Defaults to None
                kwargs['filters'] = add_parquet_AND_filter(
                    location_filter, user_filters)
                # TODO: Use logging not printing
                print(location_filter)
            else:
                print("'artifact_path' column missing from parquet file."
                      " Not filtering locations.")

        if force_parquet_dictionaries:
            # Read all columns as dictionaries except those containing 
            # floating point values
            kwargs['read_dictionary'] = [
                col.name for col in parquet_dataset.schema
                if not pa.types.is_floating(col.type)]

        # Read the parquet file
        df = pd.read_parquet(parquet_file_path, **kwargs)
        # df = parquet_dataset.read(**kwargs).to_pandas()

        if drop_superfluous_cols:
            # Drop redundant columns
            for col1, col2 in [
                ('input_draw', 'input_draw_number'),
                ('entity', 'sub_entity'),
            ]:
                if (col1 in df and col2 in df and df[col1].equals(df[col2])):
                    df.drop(columns=col2, inplace=True)
            # Drop empty columns (e.g., sub-entity)
            for col in df:
                if df[col].isna().all():
                    df.drop(columns=col, inplace=True)
        if colname_to_dtype is not None:
            # NOTE: If copy-on-write is enabled, copy keyword is ignored
            df = df.astype(
                # Must filter to avoid KeyError
                {c: dtype for c, dtype in colname_to_dtype.items() if c in df},
                copy=False
            )
        if force_pandas_categoricals:
            convert_to_categorical(
                df, exclude_cols=colname_to_dtype or (), inplace=True)
        if aggregate_seeds:
            # Use default index and value columns when aggregating
            df = vh.vph_output.operations.marginalize(df, 'random_seed')
        if assign_location:
            if all_locations_together:
                # NOTE: location_to_artifact_path is guaranteed not to
                # be None because assign_location and
                # all_locations_together are both True

                # Create a Categorical dtype with all locations
                location_dtype = pd.CategoricalDtype(
                    sorted(location_to_artifact_path.keys()), ordered=True)
                # Invert the dictionary so we can map artifact paths to
                # locations
                artifact_path_to_location = {
                    path: loc for loc, path
                    in location_to_artifact_path.items()}
                if 'artifact_path' in df:
                    df['location'] = df['artifact_path'].map(
                        artifact_path_to_location).astype(location_dtype)
                else:
                    # In case the engineers change the DataFrame format
                    # on us...
                    print("'artifact_path' column missing from DataFrame."
                          " Not assigning locations.")
            else:
                # NOTE: location_to_results_dir contains actual
                # locations as keys (not 'all') since
                # all_locations_together is False

                # Create a Categorical dtype with all locations to avoid
                # converting back to object dtype.
                location_dtype = pd.CategoricalDtype(
                    sorted(location_to_results_dir.keys()), ordered=True)
                df['location'] = location
                df['location'] = df['location'].astype(location_dtype)
        dfs.append(df)
    # TODO: Maybe if assign_location is False and all_locations_together
    # is also False (and there is more than one location?), we should
    # return a dict mapping locations to dataframes (or just a list of
    # dataframes?) instead of concatenating, since it won't be possible
    # to filter the resulting concatenated dataframe by location...
    df = pd.concat(dfs)
    return df

# # NOTE: This function is unnecessary because it does the same thing as
# # DataFrame.astype
# def convert_dtypes(df, colname_to_dtype, inplace=False):
#     if not inplace:
#         df = df.copy()
#     for colname, dtype in colname_to_dtype.items():
#         if colname in df:
#             df[colname] = df[colname].astype(dtype)

#     if not inplace:
#         return df
#     else:
#         return None

def convert_to_categorical(
        df,
        include_cols=(),
        exclude_cols=(),
        exclude_dtypes=(pd.api.types.is_numeric_dtype,
                        # TODO: For some reason,
                        # pd.api.types.is_categorical_dtype is deprecated, and
                        # they recommend using isinstance instead. So I don't
                        # know if there's a simpler way to make checking for a
                        # categorical dtype more parallel with checking for a
                        # numeric dtype.
                        lambda dtype: isinstance(dtype, pd.CategoricalDtype)),
        inplace=False
    ):
    """Convert all columns except numerical columns to categorical. This
    saves lots of memory, allowing us to load and manipulate larger
    DataFrames.
    """
    if  len(set(include_cols).intersection(exclude_cols)) != 0:
        raise ValueError("A column can't be both included and excluded!")
    if not inplace:
        df = df.copy()
    for col in df:
        if (
            col in include_cols
            or (col not in exclude_cols
                and not any(excluded_dtype(df[col].dtype)
                            for excluded_dtype in exclude_dtypes))
                # and df[col].dtype not in exclude_dtypes)
        ):
            df[col] = df[col].astype('category')
    if not inplace:
        return df
    else:
        return None
    
def add_parquet_AND_filter(new_filter, existing_filters):
    match existing_filters:
        case None:
            # No existing filters -- create a single AND group
            filters = [new_filter]
        case list([tuple((_, _, _)), *_]):
            # Existing filters consist of one AND group -- add the new filter
            filters = [new_filter, *existing_filters]
        case list([list([tuple((_, _, _)), *_]), *_]):
            # Add the filter to each AND group in the outer OR group
            filters = [[new_filter, *and_group] for and_group in existing_filters]
        case _:
            raise ValueError(f"Malformed parquet filter: {existing_filters}")
    # # Alternate version using isinstance()
    # if existing_filters is None:
    #     # No existing filters -- create a single AND group
    #     filters = [new_filter]
    # elif isinstance(existing_filters[0], tuple):
    #     # Existing filters consist of one AND group -- add the new filter
    #     filters = [new_filter, *existing_filters]
    # elif isinstance(existing_filters[0], list):
    #     # Add the filter to each AND group in the outer OR group
    #     filters = [[new_filter, *and_group] for and_group in existing_filters]
    # else:
    #     raise ValueError(f"Malformed parquet filter: {existing_filters}")
    return filters

# NOTE: Need to create ops before defining this function
def summarize_sim_data(df, age_dictionary=age_dictionary):
    """Summarize simulation data for plotting."""
    # Merge to get an age_start column for plotting
    if 'age_group' in df:
        df = df.merge(age_dictionary, on='age_group')
    # Summarize, and rename percentiles to match artifact
    summary = ops.describe(df).rename(
        columns={'2.5%': 'lower', '97.5%': 'upper'})
    return summary

def print_memory_usage(df):
    print(df.memory_usage(deep=True).sum() / 1e6, 'MB')


# Test code for loading sim output

In [ ]:
dataset_path = Path(location_to_results_dir['all']) / 'counts_new_simulants'
dataset_path

In [ ]:
dataset = pa.parquet.ParquetDataset(dataset_path)
dataset.schema

In [ ]:
new_sims = load_sim_output(
    'counts_new_simulants',
    # colname_to_dtype=None,
    raw=True,
    engine='pyarrow',
    # read_dictionary=['input_draw', 'scenario'],
    # filters=[('event_year', '==', '2025')],
    # assign_location=False,
    # aggregate_seeds=False,
)
new_sims

In [ ]:
print(new_sims.input_draw.dtype == 'int') # int64 counts as int
print(new_sims.event_year.dtype == 'int') # int16 does NOT count as int
pd.api.types.is_numeric_dtype(new_sims.event_year)

# Define plotting functions

In [ ]:
def plot_over_time_by_column(
        df,
        colname,
        ylabel='',
        title='',
        uncertainty=True,
        ax=None,
        plot_func='plot',
        style_col='auto',
        **kwargs, # Keyword args to pass to plotting function
    ):
    """Plot mean value vs. year for each value in the column `colname`,
    optionally with (lower, upper) uncertainty band.

    If `style_col` is a column name, draw one line for each pair of values
    in `colname` and `style_col`, with one color for each value of
    `colname` and one line style for each value of `style_col`. With the
    default 'auto', `style_col` is 'scenario' when df has more than one
    scenario, so that values are never averaged over scenarios.
    """
    if ax is None:
        ax = plt.gca()
    if style_col == 'auto':
        multiple_scenarios = (
            'scenario' in df and colname != 'scenario'
            and df['scenario'].nunique() > 1)
        style_col = 'scenario' if multiple_scenarios else None
    group_cols = [colname] if style_col is None else [colname, style_col]
    agg = df.groupby(group_cols + ['event_year'], observed=True)['value'].agg(['mean', lower, upper])
    linestyles = ['-', '--', ':', '-.']
    col_vals = agg.index.unique(colname)
    style_vals = [None] if style_col is None else agg.index.unique(style_col)
    ax_plotter = getattr(ax, plot_func)
    for color_num, col_val in enumerate(col_vals):
        color = f'C{color_num % 10}'
        for style_num, style_val in enumerate(style_vals):
            key = (col_val,) if style_col is None else (col_val, style_val)
            try:
                values = agg.xs(key, level=group_cols)
            except KeyError:
                # This pair of values does not occur in the data
                continue
            years = values.index
            label = f"{colname}={col_val}"
            line_kwargs = {'color': color}
            if style_col is not None:
                label += f", {style_col}={style_val}"
                line_kwargs['linestyle'] = linestyles[style_num % len(linestyles)]
            ax_plotter(years, values['mean'], label=label, **line_kwargs, **kwargs)
            if uncertainty:
                ax.fill_between(
                    years, values['lower'], values['upper'], color=color, alpha=.1)

    ax.set_xlabel('year')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    return ax

def plot_over_time_by_column_for_row_x_1(
        df, colname, row_variable, ylabel='', suptitle='', uncertainty=True,
        axes_width=12, axes_height=6,
        plot_func='plot',
        **kwargs,
    ):
    """Draw a nx1 figure with rows indexed by location, calling plot_over_time_by_column()
    for each subplot.
    """
    # Get ordered list of locations in the dataframe
    # df = csr.to_ordered_categoricals(df)
    df = convert_to_categorical(df)
    row_values = df[row_variable].unique().sort_values()
    fig, _ = plt.subplots(
        len(row_values), 1,
        figsize=(axes_width, 1 + axes_height * len(row_values)))
    for row_num, row_value in enumerate(row_values):
        plot_over_time_by_column(
            df.query(f"{row_variable}==@row_value"),
            colname,
            ylabel,
            f"{row_variable}={row_value}",
            uncertainty,
            ax=fig.axes[row_num],
            plot_func=plot_func,
            **kwargs,
        )
    fig.suptitle(suptitle, fontsize=18)
    fig.tight_layout()
    return fig

# Load treatment person-time data

In [ ]:
# 62 MB in memory for one location
person_time_treatment = load_sim_output('person_time_treatment')
print(person_time_treatment.memory_usage(deep=True).sum() / 1e6, 'MB')
person_time_treatment

# Load treatment transition counts

In [ ]:
# 55 MB in memory for one location
transition_count_treatment = load_sim_output('transition_count_treatment')
print_memory_usage(transition_count_treatment)
transition_count_treatment

In [ ]:
non_treatment_transitions = transition_count_treatment.query("scenario == 'baseline' and value > 0")
non_treatment_transitions

In [ ]:
df = (
    transition_count_treatment
    # Use the scenarios with treatment
    .query("scenario in @with_treatment_scenarios")
    .pipe(ops.marginalize, 'semester')
    .pipe(lambda df: df.join(
        cleaning.extract_transition_states(df, 'sub_entity'))
    )
    # .pipe(convert_to_categorical)
)
df .dtypes

# Transform treatment transition counts into more useful form

In [ ]:
treatment_transitions = (
    transition_count_treatment
    # Use the scenarios with treatment
    .query("scenario in @with_treatment_scenarios")
    .pipe(ops.marginalize, 'semester')
    .pipe(lambda df: df.join(
        cleaning.extract_transition_states(df, 'sub_entity'))
    )
    .pipe(convert_to_categorical)
)
print_memory_usage(treatment_transitions)
treatment_transitions

In [ ]:
treatment_transitions.to_state

# Check #1 (treatment initiation rates, aka intervention coverage)

Positive BBBM tests result in treatment initiation rates that match
the year/location specific rates from in the [treatment intervention
data
table](https://vivarium-research.readthedocs.io/en/latest/models/intervention_models/alzheimers/hypothetical_treatment.html#alzheimers-intervention-treatment-data-table)

> We will use a piecewise linear ramp-up with knots at the following (year, level) values: (2022.0, 0), (2027.0, 0), (2035.5, 30), (2100.0, 80), (2101.0, 80).
>
> This captures the CSU client’s specification that
> “30% of eligible patients initiate by 2035, with a steady increase to 80% by 2100, for all countries,” and that treatment should first be available in 2027, slowly ramping up to 30% in 2035.

## Strategy

Consider all transitions out of the `'susceptible_to_treatment'` state.
Compute the percent of transitions that go to `'waiting_for_treatment'` vs.
`'no_effect_never_treated'`. The percent that go to `'waiting_for_treatment'`
should match the location-specific rates from Lilly for each year.

## Looks like treatment still starts in 2030, not 2027

But maybe the piecewise linear function is actually correct, and it's
just getting cut off for some reason?



In [ ]:
treatment_transitions.to_state.unique()

In [ ]:
testing_eligible_age_groups = age_groups[7:11]
testing_eligible_age_groups

In [ ]:
# Testing eligible ages are 60-80, and treatment always starts 6 months
# after testing, so treatment-eligible age groups are 60-64 through
# 80-84. Note that these are the ages when people should enter one of
# the 'full_effect' states, not the 'waiting_for_treatment' state
treatment_eligible_age_groups = age_groups[7:12]
treatment_eligible_age_groups

In [ ]:
# Verify that no one outside the testing-eligible age groups starts
#  waiting for treatment
assert set(treatment_transitions.query( \
        "to_state == 'waiting_for_treatment'" \
        " and age_group not in @testing_eligible_age_groups" \
    ).value.unique()) == {0.0}, \
        "Nonzero waiting for treatment counts outside eligible age groups"

In [ ]:
treatment_transitions.query( \
        "to_state == 'waiting_for_treatment' and value > 0" \
    ).age_group.unique()

In [ ]:
# Here's more stringent way to check the same thing, verifying also that
# there *are* transitions within each of the testing-eligible age groups
assert set(treatment_transitions.query( \
        "to_state == 'waiting_for_treatment' and value > 0" \
    ).age_group.unique()) \
        == {'65_to_69', '70_to_74', '75_to_79'}, \
            "Waiting for treatment occurring in wrong age groups"

In [ ]:
treatment_transitions.query(
    "to_state == 'waiting_for_treatment' and value > 0"
).groupby('age_group', observed=True).describe()

In [ ]:
# Check that treatment starts in 2027
(
    treatment_transitions
    .astype({'event_year': 'int'})
    .query("from_state == 'susceptible_to_treatment' and value > 0")
    .groupby('location', observed=True)
    .event_year
    .min()
)

In [ ]:
initiate_vs_not = ops.ratio(
    treatment_transitions,
    treatment_transitions,
    strata=['event_year'],
    prefilter_query=(
        "from_state == 'susceptible_to_treatment'"
        # Can't pass using @ syntax since the variable is out of scope
        # when .query gets called in the .ratio function
        # f" and age_group in {treatment_eligible_age_groups}"
    ),
    numerator_broadcast=["to_state"],
    # Drop years prior to 2030 where we divide by 0 treatment transition
    # counts
    # dropna=True,
)
initiate_vs_not

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    initiate_vs_not,
    'to_state',
    'location',
    ylabel='fraction of transitions out of susceptible',
)
for ax in fig.axes:
    # Looks like this is unnecessary after converting year to int
    # ax.set_xticks(ax.get_xticks()[::10])
    ax.grid(True)

# Check #2 (completes vs. discontinues treatment)

10% of transitions to Full treatment effect status are by simulants who
discontinue treatment

## Strategy using new `treatment_duration` dataframe

For each year, compute the fraction of simulants who complete 9 months
of treatment among those who complete more than 0 months of treatment.
This should be 90%. Moreover, in this same group, for each $k\in \{1, \dotsc, 8\}$, the
fraction of those who complete $k$ months of treatment should be 10/8 =
1.25%.

Out of the total, the fraction of simulants who complete more than 0
months of treatment should (maybe) be the treatment rate for the year.

## Looks good

In [ ]:
treatment_duration = load_sim_output(
    'treatment_duration',
    colname_to_dtype = colname_to_dtype | {'sub_entity': 'int8'},
    )
treatment_duration

In [ ]:
assert set(treatment_duration.query("value > 0").scenario.unique()) == set(scenarios) - {'baseline'}, \
    "Treatments should occur in every scenario except baseline"

In [ ]:
assert set(treatment_duration.query("value > 0").age_group.unique()) \
    == {'65_to_69', '70_to_74', '75_to_79'}, \
        "Treatments occurring in wrong age groups"

In [ ]:
# Compute fraction of treatments discontinued after k months, where k =
# 1, 2, 3, ..., 9
fraction_among_treated = ops.ratio(
    treatment_duration,
    treatment_duration,
    # Stratify by year so we can plot over time
    strata=['event_year'],
    # Stratify numerator by months of treatment; denominator will be the
    # sum of all those with more than 0 months
    numerator_broadcast=['sub_entity'],
    # Filter to those who got treatment
    prefilter_query=f"sub_entity > 0 and scenario in {with_treatment_scenarios}",
).rename(columns={'sub_entity': 'months_of_treatment'})
fraction_among_treated

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    # Sub-entity is the number of months of treatment received
    fraction_among_treated.query("months_of_treatment == 9"),
    'months_of_treatment',
    'location',
    ylabel='fraction of treatments',
    suptitle='Fraction of treatments that are completed',
)
for ax in fig.axes:
    ax.grid(True)

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    fraction_among_treated.query("months_of_treatment != 9"),
    'months_of_treatment',
    'location',
    ylabel='fraction of treatments',
    suptitle='Fraction of treatments discontinued after $k$ months, where $k=1, 2, ..., 8$',
)
for ax in fig.axes:
    ax.hlines(0.1/8, 2025, 2100, 'black', '--', label='target')
    ax.legend()

## Compute total number of people treated, and number of completed vs. discontinued treatment

In [ ]:
# NOTE (2026-02-19): I originally used the following dictionary in my
# call to aggregate_categories below. Amazingly, this mostly worked as
# expected because pandas .replace can correctly interpret keys that are
# range objects! But this is not what I intended: My _ensure_iterable
# function treated the range as a singleton whereas I was (foolishly)
# expecting the ranges to be "unpacked". This caused my check for
# mutually exclusive lists to fail (it incorrectly said that the
# following dictionary had mutually exclusive lists for values), which
# led to unexpected results in the aggregation (see example series
# below). I addressed the problem by adding `range`` to the list of
# iterable types allowed by _ensure_iterable... not sure if that's the
# best solution.
d = {
        'not_treated': 0,
        'treated': range(1, 10),
        'completed_treatment': 9,
        'discontinued_treatment': range(1, 9),
    }
d_inv = {v: k for k, v in d.items()}
print(d_inv)
s = pd.Series(range(20))
s.replace(d_inv)

In [ ]:
# aggregate_categories requires mutually exclusive supercategories, and
# 'treated' overlaps 'completed_treatment' and 'discontinued_treatment',
# so aggregate each mutually exclusive set separately and concatenate
treated_vs_not = pd.concat(
    [
        ops.aggregate_categories(
            treatment_duration,
            'sub_entity',
            {
                'not_treated': 0,
                'treated': range(1, 10),
            },
        ),
        # category 0 is not in this mapping, so it is kept as is and dropped here
        ops.aggregate_categories(
            treatment_duration,
            'sub_entity',
            {
                'completed_treatment': 9,
                'discontinued_treatment': range(1, 9),
            },
        ).query("sub_entity in ['completed_treatment', 'discontinued_treatment']"),
    ],
    ignore_index=True,
)
treated_vs_not

In [ ]:
d = {
        'not_treated': 0,
        'treated': range(1, 10),
        'completed_treatment': 9,
        'discontinued_treatment': range(1, 9),
    }
vh.utils.lists_are_mutually_exclusive(d.values())

In [ ]:
vh.utils._ensure_iterable(range(1,10))

In [ ]:
supercategory_to_categories = d
category_to_supercategory = {
            category: supercategory
            for supercategory, categories
                in supercategory_to_categories.items()
            for category in vh.utils._ensure_iterable(categories)
        }

category_to_supercategory

In [ ]:
s = pd.Series(range(20))
s.replace(category_to_supercategory)

In [ ]:
d = {
        'not_treated': 0,
        'treated': range(1, 10),
        'completed_treatment': 9,
        'discontinued_treatment': range(1, 9),
    }
d_inv = {v: k for k, v in d.items()}
print(d_inv)
s = pd.Series(range(20))
s.replace(d_inv)

In [ ]:
s.replace(d_inv)

In [ ]:
type(range(1, 10))

In [ ]:
assert set(treated_vs_not.query("sub_entity == 'not_treated'").value.unique()) == {0}, \
    "Treatment durations should only be recorded among treated simulants"

## Check fraction of people who complete vs. discontinue treatment

In [ ]:
fraction_completed = ops.ratio(
    treated_vs_not,
    treated_vs_not,
    strata=['event_year'],
    numerator_broadcast='sub_entity',
    prefilter_query=(
        "sub_entity in ['completed_treatment', 'discontinued_treatment']"
        f" and age_group in {testing_eligible_age_groups}"
        f" and scenario in {with_treatment_scenarios}"
    ),
)
fraction_completed


In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    fraction_completed,
    'sub_entity',
    'location',
    ylabel='fraction of simulants',
    suptitle='Fraction of simulants who complete vs. discontinue treatment',
)
for ax in fig.axes:
    ax.grid(True)

In [ ]:
treatment_duration.query("event_year>=2050").groupby('sub_entity')['value'].sum()

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    ops.stratify(treated_vs_not.query("scenario in @with_treatment_scenarios"), ['sub_entity', 'event_year']),
    # ops.stratify(treatment_duration,)
    'sub_entity',
    'location',
    uncertainty=True,
    ylabel='number of simulants',
    suptitle='Number of simulants who get treatment vs. not',
)
for ax in fig.axes:
    ax.grid(True)
    # ax.semilogy()

In [ ]:
ops.stratify(treated_vs_not, ['sub_entity', 'event_year'])

In [ ]:
ops.stratify(treatment_duration, ['event_year'])

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    # ops.stratify(treated_vs_not, ['sub_entity', 'event_year']),
    ops.stratify(treatment_duration.query("sub_entity != 0"), ['event_year']).assign(treated='treated'),#.query(
        # "scenario in @with_treatment_scenarios"),
    'scenario',
    'location',
    uncertainty=True,
    ylabel='number of simulants',
    suptitle='Number of simulants who get treatment',
)
for ax in fig.axes:
    ax.grid(True)
    # ax.semilogy()

In [ ]:
treatment_transitions.value_counts(['from_state', 'to_state'])

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    (
        treatment_transitions
        .query(
        # "sub_entity == 'susceptible_to_treatment_to_waiting_for_treatment'"
        "from_state in ['susceptible_to_treatment', 'waiting_for_treatment', 'treatment_effect']")
        .pipe(ops.stratify, ['sub_entity', 'event_year'])
    ),
    'sub_entity',
    'location',
    uncertainty=True,
    ylabel='number of transitions',
    # suptitle='Number of simulants who get treatment vs. not',
)
for ax in fig.axes:
    # ax.semilogy()
    ax.grid(True)

In [ ]:
treatment_counts = pd.concat([treated_vs_not, treatment_transitions], ignore_index=True)
treatment_counts

In [ ]:
treatment_initiations = [
    'susceptible_to_treatment_to_waiting_for_treatment',
    'waiting_for_treatment_to_treatment_effect',
    'treated',
]

fig = plot_over_time_by_column_for_row_x_1(
    (
        treatment_counts
        .query(
            "scenario in @with_treatment_scenarios"
            " and sub_entity in @treatment_initiations"
        )
        .pipe(ops.stratify, ['sub_entity', 'event_year'])
    ),
    'sub_entity',
    'location',
    uncertainty=True,
    ylabel='number of transitions or treatments',
    # suptitle='Number of simulants who get treatment vs. not',
)
for ax in fig.axes:
    # ax.semilogy()
    ax.grid(True)

# Check #3 (durations of full vs. waning treatment)

Full/Waning durations are accurate (use person-time ratios between
states?)

## Strategy

Look at ratio of person-time in each state to the person-time in the
`'waiting_for_treatment'` state. Using the fact that 90% of simulants complete
treatment and end up in the "long" states, while 10% of simulants
discontinue treatment and end up in the "short" states, we can predict
what these ratios should be.

## Results

Things look good except for the `'waning_effect_long'` state, whose
ratio to the `'waiting_for_treatment'` state looks too low (it's the same as
for the `'full_effect_long'` state).

In [ ]:
all_tx_states = person_time_treatment.sub_entity.unique()
all_tx_states

In [ ]:
no_effect_states = [state for state in all_tx_states if 'no_effect' in state]
no_effect_states

In [ ]:
# Compute ratio of person-time in each state to the person-time in the
# "waiting_for_treatment" state
tx_person_time_ratios = ops.ratio(
    person_time_treatment,
    person_time_treatment.query("sub_entity == 'waiting_for_treatment'"),
    strata=['event_year'],
    prefilter_query=(
        f"scenario in {with_treatment_scenarios}"
        f" and event_year >= 2030" #" and age_group in {treatment_eligible_age_groups}"
        f" and sub_entity not in {no_effect_states + ['susceptible_to_treatment']}"
    ),
    numerator_broadcast=['sub_entity'],
)
tx_person_time_ratios

In [ ]:
tx_state_durations = pd.DataFrame(
    {
        # Durations from state diagram in treatment documentation
        'duration': [0.5, 5.0, 0.5, 9.0, 2.5],
        # 90% of people complete treatment and get long treatment
        # effect, 10% discontinue and get short treatment effect 
        'population_fraction': [1, 0.9, 0.1, 0.9, 0.1],
    },
    index=pd.CategoricalIndex(
        ['waiting_for_treatment', 'full_effect_long', 'full_effect_short', 'waning_effect_long', 'waning_effect_short'],
        name='sub_entity',
    )
)
# Compute the ratio of the population-weighted duration of each state to
# the duration of the "start treatment" state.
tx_state_durations['target_person_time_ratio'] = (
    tx_state_durations['duration'] * tx_state_durations['population_fraction']
    / tx_state_durations.at['waiting_for_treatment', 'duration']
)
tx_state_durations


In [ ]:
# For posting in Slack...
print(tx_state_durations.rename(
    columns={'population_fraction': 'pop_frac', 'target_person_time_ratio': 'pt_ratio'}))

In [ ]:
def plot_treatment_state_targets(tx_state_durations, column, ax):
    """Plot horizontal lines for values in one column of the treatment
    state durations table.
    """
    existing_lines = ax.get_lines()
    if len(existing_lines) == 0:
        # Axes.hlines plots all lines in a single color by default, so if you
        # want different colors, you have to explicitly cycle through them 
        prop_cycle = plt.rcParams['axes.prop_cycle']
        colors = prop_cycle.by_key()['color']

    for i, state in enumerate(tx_state_durations.index):
        # Find color corresponding to state if there already exists a
        # line for that state
        for line in existing_lines:
            if state in line.get_label():
                color = line.get_color()
                break
        # If there were no existing lines, go to next color in cycle
        if len(existing_lines) == 0:
            color = colors[i % len(colors)]
        ax.hlines(tx_state_durations.at[state, column], 2030, 2100, color, '--', label=f'{state} (target)')
    return ax

fig, ax = plt.subplots(figsize=(12, 6))

ax = plot_treatment_state_targets(tx_state_durations, 'target_person_time_ratio', ax)
ax.set(
    xlabel='year',
    ylabel='person-time ratio',
    title='Target ratios of Tx state person-time to "waiting_for_treatment" person-time',
)
ax.grid(True)
ax.legend()
ax.semilogy()
plt.show()

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    tx_person_time_ratios,
    'sub_entity',
    'location',
    ylabel='person-time ratio',
    suptitle='Observed ratios of Tx state person-time to "waiting_for_treatment" person-time',
    axes_width=9,
    axes_height=4,
)
for ax in fig.axes:
    ax = plot_treatment_state_targets(tx_state_durations, 'target_person_time_ratio', ax)
    # ax.set_xticks(ax.get_xticks()[::10])
    ax.grid(True)
    ax.legend()
    ax.semilogy()
plt.show()

## Look at actual person-time instead of the ratio

In [ ]:
ops.describe(
    ops.stratify(
        person_time_treatment.query(
            "value > 0 and "
            "(sub_entity == 'waiting_for_treatment' or "
            " sub_entity.str.endswith('effect_long')) and event_year > 2050"
            # " and location == 'Spain'"
        ),
        ['event_year', 'sub_entity']
    )
)

In [ ]:
9721.296427 / 1129.481287

In [ ]:
13995.099904 / 1129.481287

# Check #4 (duration of start treatment/waiting for treatment)

“In treatment/waiting for treatment” duration is accurate (use
person-time ratios between states?)

This duration is supposed to be 6 months for all simulants.

## Strategy

To compute the average duration of each state, divide the total
person-time of that state by the number of simulants entering that
state, i.e., the the sum of transition counts where it's the `to_state`.

Alternatively, compute the average duration by dividing the total
person-time by the number of simulants *exiting* the state, i.e., the
sum of transition counts where it's the `from_state`.

## Results

I'm not sure how closely either of these strategies should match the
actual duration... Neither takes deaths into account, and I stratified
the calculations by year, which seems weird because some of the
durations are much longer than a year.

Nevertheless, the observed average durations for 4 out of 5 of the
states eventually converge to a value pretty close to the specified
duration, using either of the above two methods. The fifth state,
`'waning_effect_long'`, converges to different values in each of the
above two methods, neither of which is particularly close to the target
duration of 9 years. We think this is probably due to mortality.

In [ ]:
person_time_treatment

In [ ]:
treatment_transitions.query("value>0")

In [ ]:
tx_state_mean_durations_incoming = ops.ratio(
    person_time_treatment.assign(to_state=lambda df: df['sub_entity']),
    treatment_transitions.query("value>0"),
    strata=['event_year', 'to_state'],
    prefilter_query=(
        f"scenario in {with_treatment_scenarios} and event_year >= 2030"
    )
)
tx_state_mean_durations_incoming

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax = plot_treatment_state_targets(tx_state_durations, 'duration', ax)
ax.set(
    xlabel='year',
    ylabel='person-time ratio',
    title='Target durations of Tx states',
)
ax.grid(True)
ax.legend()
ax.set(yticks=range(0,10))
# ax.semilogy()
plt.show()

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    tx_state_mean_durations_incoming,
    'to_state',
    'location',
    ylabel='average state duration (years)',
    suptitle="Mean duration of Tx state among those entering state",
    # axes_width=9,
    # axes_height=4,
)
for ax in fig.axes:
    ax = plot_treatment_state_targets(tx_state_durations, 'duration', ax)
    ax.legend()
    ax.grid(True)
    ax.semilogy()
plt.show()

In [ ]:
tx_state_mean_durations_outgoing = ops.ratio(
    person_time_treatment.assign(from_state=lambda df: df['sub_entity']),
    treatment_transitions.query("value>0"),
    strata=['event_year', 'from_state'],
    prefilter_query=(
        f"scenario in {with_treatment_scenarios} and event_year >= 2030"
    )
)
tx_state_mean_durations_outgoing

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax = plot_treatment_state_targets(tx_state_durations, 'duration', ax)
ax.set(
    xlabel='year',
    ylabel='person-time ratio',
    title='Target durations of Tx states',
)
ax.grid(True)
ax.legend()
ax.set(yticks=range(0,10))
ax.semilogy()
plt.show()

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    tx_state_mean_durations_outgoing,
    'from_state',
    'location',
    ylabel='average state duration (years)',
    suptitle="Mean duration of Tx state among those exiting state",
)
for ax in fig.axes:
    ax = plot_treatment_state_targets(tx_state_durations, 'duration', ax)
    ax.legend()
    ax.grid(True)
    ax.semilogy()
plt.show()

# Check #6 (hazard ratio)

Check hazard ratios for simulants who begin treatment and those who
transition to *No treatment effect* (Check hazard ratio of treated to
not treated for hazard rate from BBBM to MCI)

## I think I can't check this in the outputs because neither cause transitions nor cause state person-time is stratified by treatment status

In [ ]:
person_time_cause_states = load_sim_output(
    'person_time_alzheimers_disease_and_other_dementias',
)
print_memory_usage(person_time_cause_states)
person_time_cause_states

In [ ]:
transition_count_cause_states = (
    load_sim_output(
        'transition_count_alzheimers_disease_and_other_dementias',
    )
    # Display memory usage of loaded dataframe before adding columns
    .pipe(lambda df: print_memory_usage(df) or df)
    # Add columns for from and to state
    .pipe(lambda df: df.join(
        cleaning.extract_transition_states(df, 'sub_entity'))
    )
    # Display memory usage of loaded dataframe after adding columns
    .pipe(lambda df: print_memory_usage(df) or df)
    # Convert the new columns to Categoricals
    .pipe(convert_to_categorical)
)
print_memory_usage(transition_count_cause_states)
transition_count_cause_states

In [ ]:
incidence_rates_by_treatment = ops.ratio(
    transition_count_cause_states,
    person_time_cause_states.rename(columns={'sub_entity': 'from_state'}),
    strata=['event_year', 'from_state', 'treatment'],
    numerator_broadcast='to_state', # doesn't actually broadcast, just records it
    measure='transition_rate',
    dropna=True,
)
incidence_rates_by_treatment

## Investigate infinite transition rates between disease states

My plotting code below initially failed with `FloatingPointError:
invalid value encountered in subtract` in the `lower` function, due to
infinities in the transition rates data.

According to the engineers, we may be getting 0 person-time in some
strata where counts are nonzero because person-time is recorded at the
beginning of a time step, while transition counts are recorded at the
end. So for example, if someone starts in preclinical, in the "waiting
for treatment" state, then transitions to MCI and to "full effect long"
on the same time step, then they would have their person-time recorded
in the (preclinical, waiting for treatment) stratum, but their disease
transition recorded in the (preclinical- > MCI, full effect long)
stratum.

The engineers said there was a bugfix in model 7.2 that *might* address
this problem, so I'll try this again there...

In [ ]:
# 2026-02-05: Why do I have to explicitly put this in a context manager all of a sudden?
with np.errstate(invalid='ignore'):
    print(incidence_rates_by_treatment.value.describe())

In [ ]:
inf = np.isinf(incidence_rates_by_treatment.value)
print(inf.sum(), "rows with infinite values")
incidence_rates_by_treatment.loc[inf]#.query("from_state=='alzheimers_blood_based_biomarker_state'")

## Plot disease transition rates stratified by treatment status, filtering out $\infty$

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    # NOTE: Without filtering infinities out, the `lower` method fails
    # with "FloatingPointError: invalid value encountered in subtract"
    incidence_rates_by_treatment.loc[~inf].query("from_state=='alzheimers_blood_based_biomarker_state' and scenario in @with_treatment_scenarios"),
    'treatment',
    'location',
    ylabel='MCI incidence rate',
    suptitle='MCI incidence rate stratified by treatment status',
    # uncertainty=False,
)
for ax in fig.axes:
    ax.grid(True)
    # ax.semilogy()
plt.show()

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    # NOTE: Without filtering infinities out, the `lower` method fails
    # with "FloatingPointError: invalid value encountered in subtract"
    incidence_rates_by_treatment.loc[~inf].query("from_state=='alzheimers_mild_cognitive_impairment_state' and scenario in @with_treatment_scenarios"),
    'treatment',
    'location',
    ylabel='AD dementia incidence rate',
    suptitle='AD dementia incidence rate stratified by treatment status',
    # uncertainty=False,
)
for ax in fig.axes:
    ax.grid(True)
    # ax.semilogy()
plt.show()

## Compute hazard ratio

In [ ]:
hazard_ratio = ops.ratio(
    incidence_rates_by_treatment,
    incidence_rates_by_treatment.query("treatment=='susceptible_to_treatment'"),
    strata=['event_year', 'from_state'],
    numerator_broadcast=['treatment'],
    # prefilter_query="from_state=='alzheimers_blood_based_biomarker_state'",
)
hazard_ratio

In [ ]:
with np.errstate(invalid='ignore'):
    print(hazard_ratio.value.describe())

In [ ]:
inf_ratio = np.isinf(hazard_ratio.value)
print(inf_ratio.sum(), "rows with infinite values")

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    # NOTE: Without filtering infinities out, the `lower` method fails
    # with "FloatingPointError: invalid value encountered in subtract"
    hazard_ratio.loc[~inf_ratio].query(
        "scenario in @with_treatment_scenarios"
        " and from_state=='alzheimers_blood_based_biomarker_state'"
        " and event_year > 2035" # Avoid wild spikes in early years
        ),
    'treatment',
    'location',
    ylabel='MCI incidence rate ratio',
    suptitle='MCI incidence rate ratio relative to "susceptible_to_treatment"',
    # uncertainty=False,
)
for ax in fig.axes:
    ax.grid(True)
    # ax.semilogy()
plt.show()

In [ ]:
fig = plot_over_time_by_column_for_row_x_1(
    # NOTE: Without filtering infinities out, the `lower` method fails
    # with "FloatingPointError: invalid value encountered in subtract"
    hazard_ratio.loc[~inf_ratio].query(
        "scenario in @with_treatment_scenarios and event_year > 2035"
        " and from_state=='alzheimers_mild_cognitive_impairment_state'"
        ),
    'treatment',
    'location',
    ylabel='AD dementia incidence rate ratio',
    suptitle='AD dementia incidence rate ratio relative to "susceptible_to_treatment"',
    # uncertainty=False,
)
for ax in fig.axes:
    ax.grid(True)
    # ax.semilogy()
plt.show()

In [ ]:
# Why are the hazard ratios above 0.5?
# Maybe it's selection bias due to age?
hazard_ratio.loc[~inf_ratio].query(
    "scenario in @with_treatment_scenarios"# and event_year > 2035"
    " and from_state=='alzheimers_blood_based_biomarker_state'"
    " and treatment == 'treatment_effect'"
    " and event_year >= 2040"
    ).pipe(ops.describe)

# Results!

# Load deaths and plot averted deaths by scenario

In [ ]:
deaths = load_sim_output('deaths')
print_memory_usage(deaths)
deaths

In [ ]:
# NOTE: I had to rework some code in my vph_output.operators.difference
# function to prevent this code from converting Categoricals back to
# strings and making the memory explode...
averted_deaths = ops.averted(deaths, baseline_scenario='baseline')
print_memory_usage(averted_deaths)
# convert_dtypes(averted_deaths, colname_to_dtype=colname_to_categorical_dtype, inplace=True)
# print_memory_usage(averted_deaths)
# convert_to_categorical(averted_deaths, inplace=True)
# print_memory_usage(averted_deaths)
averted_deaths

In [ ]:
# Print current time to easily check if notebook ran all the way through
# vs. hitting an error somewhere above
import datetime
print(datetime.datetime.now())